# 08: Multi-Layer Networks - Creating Depth

## Beyond the Single Perceptron

A single perceptron can only learn linear boundaries. To learn complex patterns like XOR, we need to **stack layers** of neurons. This is the birth of the neural network!

### The Web Dev Analogy

Think of it like middleware in Express.js:
- Each layer transforms the data
- Each layer adds a level of abstraction
- The final layer makes the decision

```javascript
app.use(parseInput)    // Layer 1: raw → features
app.use(extractPatterns) // Layer 2: features → patterns
app.use(makeDecision)  // Layer 3: patterns → output
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Ready to build deep networks! 🧠")

## 1. Why We Need Multiple Layers

Remember XOR? Let's see how adding a layer solves it.

In [ ]:
# XOR data
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])

print("XOR Problem:")
for x, y in zip(X_xor, y_xor):
    print(f"  {x} → {y[0]}")

# Visualize
plt.figure(figsize=(6, 5))
for xi, yi in zip(X_xor, y_xor):
    color = 'green' if yi[0] == 1 else 'red'
    plt.scatter(xi[0], xi[1], c=color, s=300, edgecolors='black')
    
plt.xlabel('Input 1')
plt.ylabel('Input 2')
plt.title('XOR: Cannot be separated by a single line')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Activation Functions

For multi-layer networks, we need **smooth** activation functions so we can compute gradients:

In [ ]:
def sigmoid(z):
    """Sigmoid activation: smooth step function."""
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    """Derivative of sigmoid for backpropagation."""
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    """ReLU activation: max(0, z)."""
    return np.maximum(0, z)

def relu_derivative(z):
    """Derivative of ReLU."""
    return (z > 0).astype(float)

def tanh(z):
    """Tanh activation: outputs -1 to 1."""
    return np.tanh(z)

def tanh_derivative(z):
    """Derivative of tanh."""
    return 1 - np.tanh(z)**2

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
z = np.linspace(-5, 5, 200)

activations = [
    (sigmoid, sigmoid_derivative, 'Sigmoid'),
    (relu, relu_derivative, 'ReLU'),
    (tanh, tanh_derivative, 'Tanh'),
]

for ax, (func, deriv, name) in zip(axes, activations):
    ax.plot(z, func(z), 'b-', linewidth=2, label='f(z)')
    ax.plot(z, deriv(z), 'r--', linewidth=2, label="f'(z)")
    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel('z')
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 ReLU is most popular today - simple and works well!")

## 3. Building a 2-Layer Network

Let's build a network with:
- **Input layer**: 2 neurons (our inputs)
- **Hidden layer**: 4 neurons (learns intermediate features)
- **Output layer**: 1 neuron (final prediction)

In [ ]:
class NeuralNetwork:
    """
    A simple 2-layer neural network.
    
    Architecture: Input → Hidden (with activation) → Output (with sigmoid)
    """
    
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.5):
        self.lr = learning_rate
        
        # Initialize weights with small random values
        # Xavier initialization for better convergence
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))
        
        self.history = {'loss': []}
    
    def forward(self, X):
        """
        Forward pass: compute predictions.
        
        X → [Linear → Activation] → [Linear → Sigmoid] → Output
        """
        # Hidden layer
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)  # Hidden activations
        
        # Output layer
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = sigmoid(self.z2)  # Output (probability)
        
        return self.a2
    
    def backward(self, X, y):
        """
        Backward pass: compute gradients.
        
        This is backpropagation - we'll explain it in detail next notebook!
        """
        m = X.shape[0]  # Number of samples
        
        # Output layer gradients
        dz2 = self.a2 - y  # Error at output
        dW2 = (1/m) * np.dot(self.a1.T, dz2)
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)
        
        # Hidden layer gradients
        dz1 = np.dot(dz2, self.W2.T) * sigmoid_derivative(self.z1)
        dW1 = (1/m) * np.dot(X.T, dz1)
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)
        
        # Update weights
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
    
    def compute_loss(self, y_true, y_pred):
        """Binary cross-entropy loss."""
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def train(self, X, y, epochs=10000, verbose=True):
        """Train the network."""
        for epoch in range(epochs):
            # Forward pass
            y_pred = self.forward(X)
            
            # Compute loss
            loss = self.compute_loss(y, y_pred)
            self.history['loss'].append(loss)
            
            # Backward pass
            self.backward(X, y)
            
            if verbose and epoch % 2000 == 0:
                acc = np.mean((y_pred > 0.5) == y)
                print(f"Epoch {epoch:5d}: Loss = {loss:.4f}, Accuracy = {acc:.2%}")
        
        return self
    
    def predict(self, X):
        """Make predictions."""
        return (self.forward(X) > 0.5).astype(int)

In [ ]:
# Train on XOR!
print("Training neural network on XOR...\n")

nn = NeuralNetwork(input_size=2, hidden_size=4, output_size=1, learning_rate=1.0)
nn.train(X_xor, y_xor, epochs=10000)

# Test
print("\n" + "="*40)
print("Results:")
predictions = nn.predict(X_xor)
for x, y_true, y_pred in zip(X_xor, y_xor, predictions):
    status = "✓" if y_pred[0] == y_true[0] else "✗"
    print(f"  {x} → {y_pred[0]} (expected {y_true[0]}) {status}")

print(f"\n🎉 The neural network learned XOR!")

In [ ]:
# Visualize learning and decision boundary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(nn.history['loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].set_yscale('log')

# Decision boundary
ax = axes[1]
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
Z = nn.forward(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdYlGn', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

for xi, yi in zip(X_xor, y_xor):
    color = 'green' if yi[0] == 1 else 'red'
    ax.scatter(xi[0], xi[1], c=color, s=200, edgecolors='black', zorder=3)

ax.set_xlabel('Input 1')
ax.set_ylabel('Input 2')
ax.set_title('Decision Boundary (Non-Linear!)')

plt.tight_layout()
plt.show()

print("💡 The hidden layer creates a NON-LINEAR decision boundary!")

## 4. What the Hidden Layer Learns

Let's peek inside to see what the hidden neurons learned:

In [ ]:
# Get hidden layer activations
hidden_activations = sigmoid(np.dot(X_xor, nn.W1) + nn.b1)

print("Hidden Layer Activations:")
print("Input      → Hidden Layer Activations")
print("-" * 50)
for x, h in zip(X_xor, hidden_activations):
    print(f"{x} → {h.round(3)}")

print("\n💡 The hidden layer transforms the inputs into a new representation")
print("   where XOR IS linearly separable!")

In [ ]:
# Visualize the transformation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original space
ax = axes[0]
for xi, yi in zip(X_xor, y_xor):
    color = 'green' if yi[0] == 1 else 'red'
    ax.scatter(xi[0], xi[1], c=color, s=200, edgecolors='black')
ax.set_xlabel('Input 1')
ax.set_ylabel('Input 2')
ax.set_title('Original Input Space')
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)

# Transformed space (using first 2 hidden neurons)
ax = axes[1]
for hi, yi in zip(hidden_activations, y_xor):
    color = 'green' if yi[0] == 1 else 'red'
    ax.scatter(hi[0], hi[1], c=color, s=200, edgecolors='black')
ax.set_xlabel('Hidden Neuron 1')
ax.set_ylabel('Hidden Neuron 2')
ax.set_title('Hidden Layer Space (Transformed!)')

# Draw a possible separating line
ax.plot([0, 1], [1, 0], 'k--', alpha=0.5, label='Now separable!')
ax.legend()

plt.tight_layout()
plt.show()

print("🔑 The hidden layer learned to TRANSFORM the data into a space")
print("   where the classes ARE linearly separable!")

## 5. A Harder Problem: Circles

Let's try a more complex non-linear pattern:

In [ ]:
# Generate circle data
np.random.seed(42)
n_samples = 200

# Inner circle (class 0)
r_inner = np.random.uniform(0, 0.5, n_samples // 2)
theta_inner = np.random.uniform(0, 2 * np.pi, n_samples // 2)
X_inner = np.c_[r_inner * np.cos(theta_inner), r_inner * np.sin(theta_inner)]

# Outer ring (class 1)
r_outer = np.random.uniform(0.7, 1.0, n_samples // 2)
theta_outer = np.random.uniform(0, 2 * np.pi, n_samples // 2)
X_outer = np.c_[r_outer * np.cos(theta_outer), r_outer * np.sin(theta_outer)]

X_circles = np.vstack([X_inner, X_outer])
y_circles = np.array([[0]] * (n_samples // 2) + [[1]] * (n_samples // 2))

# Shuffle
shuffle_idx = np.random.permutation(n_samples)
X_circles = X_circles[shuffle_idx]
y_circles = y_circles[shuffle_idx]

# Plot
plt.figure(figsize=(6, 6))
plt.scatter(X_circles[y_circles.ravel()==0, 0], X_circles[y_circles.ravel()==0, 1], 
            c='red', alpha=0.6, label='Class 0 (inner)')
plt.scatter(X_circles[y_circles.ravel()==1, 0], X_circles[y_circles.ravel()==1, 1], 
            c='green', alpha=0.6, label='Class 1 (outer)')
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Circle Dataset - Definitely Not Linearly Separable!')
plt.legend()
plt.axis('equal')
plt.show()

In [ ]:
# Train on circles
print("Training on circle data...\n")

nn_circles = NeuralNetwork(input_size=2, hidden_size=8, output_size=1, learning_rate=2.0)
nn_circles.train(X_circles, y_circles, epochs=10000)

# Evaluate
predictions = nn_circles.predict(X_circles)
accuracy = np.mean(predictions == y_circles)
print(f"\nFinal accuracy: {accuracy:.2%}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(nn_circles.history['loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

# Decision boundary
ax = axes[1]
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
Z = nn_circles.forward(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdYlGn', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

ax.scatter(X_circles[y_circles.ravel()==0, 0], X_circles[y_circles.ravel()==0, 1], 
           c='red', alpha=0.6, edgecolors='black', linewidth=0.5)
ax.scatter(X_circles[y_circles.ravel()==1, 0], X_circles[y_circles.ravel()==1, 1], 
           c='green', alpha=0.6, edgecolors='black', linewidth=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Learned Circular Decision Boundary!')
ax.axis('equal')

plt.tight_layout()
plt.show()

print("🎉 The network learned a circular boundary!")

## 6. Network Architecture Choices

How do we choose the number of layers and neurons?

In [ ]:
# Compare different hidden layer sizes
hidden_sizes = [2, 4, 8, 16]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, h_size in zip(axes, hidden_sizes):
    # Train
    np.random.seed(42)
    nn_test = NeuralNetwork(2, h_size, 1, learning_rate=2.0)
    nn_test.train(X_circles, y_circles, epochs=5000, verbose=False)
    
    # Accuracy
    acc = np.mean(nn_test.predict(X_circles) == y_circles)
    
    # Plot decision boundary
    xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 100), np.linspace(-1.5, 1.5, 100))
    Z = nn_test.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 11), cmap='RdYlGn', alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    ax.scatter(X_circles[y_circles.ravel()==0, 0], X_circles[y_circles.ravel()==0, 1], 
               c='red', alpha=0.5, s=20)
    ax.scatter(X_circles[y_circles.ravel()==1, 0], X_circles[y_circles.ravel()==1, 1], 
               c='green', alpha=0.5, s=20)
    ax.set_title(f'{h_size} Hidden Neurons (Acc: {acc:.1%})')
    ax.axis('equal')

plt.tight_layout()
plt.show()

print("💡 More neurons = more capacity to learn complex boundaries")
print("   But too many can lead to overfitting!")

## 📝 Check Your Understanding

1. Why do we need hidden layers?
2. What does the hidden layer "learn"?
3. Why do we need non-linear activation functions?
4. What happens if we use linear activations only?
5. How do we choose the number of hidden neurons?

## 🎯 Summary

You learned:
- **Hidden layers** transform data into a learnable representation
- **Activation functions** add non-linearity (essential!)
- **More neurons** = more capacity to learn complex patterns
- The network learns to **transform** the space so classes become separable

**Next up**: Understanding backpropagation - how networks learn! →